# Loading & Visualising Financial Data in Google Colab — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press

---
**Learning Objectives:**
- Download equity and macro data with `yfinance` and `pandas-datareader` (FRED)
- Inspect a fresh DataFrame the right way: `.head()`, `.info()`, `.isnull().sum()`
- Recognise and handle real-world data issues: missing values, mixed calendars, survivorship bias, currencies
- Turn a default matplotlib chart into publication quality with five reusable tricks
- Wrap repetitive logic into a `load_panel()` function — a template you reuse all semester

> Run each cell with **Shift+Enter**. This notebook accompanies the lecture slides.

## Step 0 — Install & Import

In [ ]:
!pip install yfinance pandas-datareader seaborn --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# FHNW corporate palette
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'

# Global plotting defaults — apply once, every chart benefits
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.25, 'font.size':11,
})
print('✓ Libraries loaded.')

---
# Part 1 — Single-Asset Download & Inspect

**Rule #1:** Whenever you load a fresh DataFrame, *inspect it before you trust it*.

### 1.1 Download Apple with one line

In [ ]:
df = yf.download('AAPL', start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)

# What just happened?
# - 'AAPL'        : ticker symbol (US equity)
# - auto_adjust  : applies splits + dividends → 'total return' price series
# - progress=False : silences the download bar (cleaner output)

print(f'Type:  {type(df).__name__}')
print(f'Shape: {df.shape}  (rows × columns)')
print(f'Range: {df.index[0].date()} → {df.index[-1].date()}')

### 1.2 Always inspect first — three commands you'll use forever

In [ ]:
# 1) head() — first 5 rows.  Sanity check on values and column names.
df.head()

In [ ]:
# 2) info() — dtypes, memory, NaN summary in one shot.
df.info()

In [ ]:
# 3) isnull().sum() — explicit count of missing values per column.
df.isnull().sum()

### 1.3 A quick first plot — does it look right?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df['Close'], color='black', lw=1.4)
ax.set_title('AAPL — Adjusted Close, 2019–2024', loc='left',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Price (USD)')
plt.tight_layout(); plt.show()

### 1.4 Returns & a 1-line statistical summary

In [ ]:
ret = df['Close'].pct_change().dropna() * 100  # in %
print('Daily returns — descriptive statistics (%):')
print(ret.describe().round(4))

---
# Part 2 — Cleaning Real-World Data

Four things go wrong in practice. Recognise them, fix them.

### 2.1 Missing values
Different markets have different holidays. When you align a US stock with a Swiss stock with crypto, you get NaNs.

In [ ]:
# Download three assets with very different trading calendars
tickers = ['AAPL', 'NESN.SW', 'BTC-USD']
raw = yf.download(tickers, start='2024-01-01', end='2024-01-15',
                   auto_adjust=True, progress=False)
prices = raw['Close'].copy()

print('First two weeks of 2024 — NaNs everywhere because the calendars differ:')
prices.head(10)

In [ ]:
# Three standard fixes — pick the one that matches your question
fix_drop  = prices.dropna()                 # only days where ALL three traded
fix_ffill = prices.ffill().dropna()          # forward-fill the gaps

print(f'Original:        {len(prices):>3} rows')
print(f'After dropna():  {len(fix_drop):>3} rows  (strict — common dates only)')
print(f'After ffill():   {len(fix_ffill):>3} rows  (carry last value forward)')

### 2.2 Different trading calendars — how many days does each market actually trade?

In [ ]:
# One full year tells the story:
year = yf.download(['AAPL','NESN.SW','BTC-USD'], start='2023-01-01',
                    end='2024-01-01', auto_adjust=True, progress=False)['Close']
for col in year.columns:
    n = year[col].dropna().shape[0]
    print(f'{col:<10}  {n:>3} trading days in 2023')

### 2.3 Survivorship bias — the silent killer
`yfinance` returns only firms still listed today. Failed firms (Lehman, Enron, WeWork…) silently disappear from the universe.
Long-run backtests on a yfinance universe will look better than reality.
**Fix:** for academic work, use point-in-time index constituents (CRSP, Compustat). For class exercises, just be aware.

### 2.4 Currency mix — never combine prices in different currencies directly
Apple is in USD. Nestlé is in CHF. Bitcoin is in USD.
*Combining returns* is fine (returns are unit-free for log-returns and approximately unit-free for small simple-returns).
*Combining prices* is meaningless. **Rule:** always work in returns when mixing currencies.

---
# Part 3 — Macro Data from FRED

FRED (Federal Reserve Economic Data, St. Louis Fed) has thousands of macro series — rates, inflation, unemployment, FX. Free, reliable, and one line of code through `pandas-datareader`.

### 3.1 The 10-year US Treasury yield

In [ ]:
# DGS10 = '10-Year Treasury Constant Maturity Rate'
y10 = web.DataReader('DGS10', 'fred', '2019-01-01', '2024-12-31')
print(y10.tail())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y10.index, y10['DGS10'], color=RED, lw=1.4)
ax.set_title('US 10-Year Treasury Yield, 2019–2024 (FRED: DGS10)',
             loc='left', fontweight='bold', fontsize=13)
ax.set_ylabel('Yield (%)')
plt.tight_layout(); plt.show()

### 3.2 Useful FRED tickers to remember

| Code | Series |
|------|--------|
| `DGS10` | 10-year Treasury yield |
| `DGS2`  | 2-year Treasury yield |
| `CPIAUCSL` | US Consumer Price Index (monthly) |
| `UNRATE` | US unemployment rate (monthly) |
| `DEXSZUS` | CHF / USD exchange rate |
| `VIXCLS` | CBOE Volatility Index (VIX) |
| `MORTGAGE30US` | 30-year fixed mortgage rate |

---
# Part 4 — Publication-Quality Charts

The same data plotted *poorly* is forgettable. Plotted *well*, it tells a story. Five tricks make 90% of the difference.

### 4.1 Re-download Apple cleanly

In [ ]:
df = yf.download('AAPL', start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)
sp = yf.download('^GSPC', start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)
aapl  = df['Close'].squeeze()
sp500 = sp['Close'].squeeze()

# Rescale S&P 500 to start at the same value as AAPL — easier visual comparison
sp500_norm = sp500 / sp500.iloc[0] * aapl.iloc[0]
print(f'AAPL: {len(aapl)} obs  •  ^GSPC: {len(sp500)} obs')

### 4.2 Before — matplotlib defaults

In [ ]:
# Reset rcParams briefly so we see the *raw* default chart
with plt.rc_context({'axes.spines.top':True, 'axes.spines.right':True,
                      'axes.grid':False}):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(aapl.index, aapl.values, label='AAPL')
    ax.plot(aapl.index, sp500_norm.values, label='S&P 500 (rescaled)')
    ax.legend(loc='upper left')   # default location — risks covering data
    ax.set_title('AAPL')
    plt.tight_layout(); plt.show()

print('Issues: spines everywhere, default colors, no annotations, generic title, legend can cover data.')

### 4.3 After — publication quality (the five tricks in one cell)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))

# Trick 2: brand palette — use 2-3 colors consistently
ax.plot(aapl.index, aapl.values, color='black', lw=1.6, label='AAPL')
ax.plot(aapl.index, sp500_norm.values, color=ORANGE, lw=1.4,
        label='S&P 500 (rescaled)')

# Trick 4: annotate the key event
covid_low = aapl[(aapl.index >= '2020-02-20') & (aapl.index <= '2020-04-30')].idxmin()
ax.annotate('COVID crash',
            xy=(covid_low, aapl.loc[covid_low]),
            xytext=(pd.Timestamp('2019-04-01'), aapl.loc[covid_low] - 25),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.2),
            fontsize=11, color=RED, fontweight='bold')

# Trick 5: legend OUTSIDE the plot area — never covers data
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=2, frameon=False, fontsize=11)

# Trick 3: clear left-aligned title with units in y-label
ax.set_title('Apple Inc. — Adjusted Closing Price, 2019–2024',
             loc='left', fontweight='bold', fontsize=13, pad=28)
ax.set_ylabel('Price (USD)')
ax.set_xlabel('')

# Trick 1 (spines off) is already in our global rcParams
plt.tight_layout(); plt.show()

### 4.4 The five tricks summarised

| # | Trick | One-line code |
|---|-------|---------------|
| 1 | **Spines off** — global, set once | `plt.rcParams['axes.spines.top']=False` |
| 2 | **Brand palette** — 2–3 colors max | use `'black'`, `ORANGE`, `RED` |
| 3 | **Clear title + units** | `ax.set_title(..., loc='left')`; `'Price (USD)'` |
| 4 | **Annotate events** | `ax.annotate('COVID', xy=..., xytext=..., arrowprops=...)` |
| 5 | **Legend outside** | `ax.legend(bbox_to_anchor=(0.5, 1.02))` |

---
# Part 5 — The `load_panel()` Template

You will compute panel returns dozens of times this semester. Wrap it in a function once.

### 5.1 The function

In [ ]:
def load_panel(tickers, start, end):
    """Download adjusted closes for a list of tickers and return a clean
    daily-return panel (DataFrame, dates × tickers)."""
    raw = yf.download(tickers, start=start, end=end,
                      auto_adjust=True, progress=False)
    prices = raw['Close'].dropna(how='all')   # drop fully empty dates
    return prices.pct_change().dropna()

print(load_panel.__doc__)

### 5.2 Mini-Case: a 4-Asset Dashboard — step by step

We'll build a small portfolio dashboard from scratch, walking through every step. By the end, you'll have a chart you could put in a report.

**Step 1 — define your universe.** Three large-cap US stocks plus the S&P 500 as a benchmark.

In [ ]:
tickers = ['AAPL', 'MSFT', 'JPM', '^GSPC']
ret = load_panel(tickers, '2019-01-01', '2024-12-31')

# Quick inspect — shape and head should make sense
print(f'Shape: {ret.shape}   ({ret.shape[0]} trading days × {ret.shape[1]} assets)')
ret.head()

**Step 2 — sanity-check the columns.** `yf.download` returns a MultiIndex; our function flattened it. Make sure the column names are what we expect.

In [ ]:
print('Columns in the returns panel:', list(ret.columns))
# Rename '^GSPC' to a friendlier label for the chart
ret = ret.rename(columns={'^GSPC': 'S&P 500'})
print('After rename:', list(ret.columns))

**Step 3 — compute cumulative returns.** Daily returns are noisy; the cumulative path tells the story. We use the standard compounding formula:
$$\text{Cumulative}_t = \prod_{s=1}^{t}(1 + r_s) - 1$$

In [ ]:
cum = (1 + ret).cumprod() - 1
print('First 3 days of cumulative returns:')
print((cum.head(3) * 100).round(3))
print('\nLast row — total cumulative performance over the period:')
print((cum.iloc[-1] * 100).round(2).sort_values(ascending=False))

**Step 4 — plot the dashboard with publication-quality styling.** Notice we apply the five tricks from Part 4 here too.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

colors = {'AAPL': 'black', 'MSFT': ORANGE, 'JPM': BLUE, 'S&P 500': RED}
for col in cum.columns:
    ax.plot(cum.index, cum[col] * 100, color=colors[col], lw=1.6, label=col)

ax.axhline(0, color=GREY, lw=0.8)

# Legend outside — never covers the lines
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=4, frameon=False, fontsize=11)

ax.set_title('4-Asset Mini-Case — Cumulative Returns 2019–2024',
             loc='left', fontweight='bold', fontsize=13, pad=28)
ax.set_ylabel('Cumulative return (%)')
plt.tight_layout(); plt.show()

**Step 5 — extract the key numbers for the report.** A chart on its own is incomplete. Add summary stats.

In [ ]:
TRADING_DAYS = 252
summary = pd.DataFrame({
    'Total Return %':   (cum.iloc[-1] * 100).round(2),
    'Annualised Vol %': (ret.std() * np.sqrt(TRADING_DAYS) * 100).round(2),
    'Worst Day %':      (ret.min() * 100).round(2),
    'Best Day %':       (ret.max() * 100).round(2),
})
print('4-Asset Mini-Case — Summary Statistics 2019–2024:')
print(summary)

**Why the function matters.** Want a different universe? Just change the inputs:
```python
swiss = load_panel(['NESN.SW','NOVN.SW','ROG.SW','^SSMI'], '2020-01-01', '2024-12-31')
crypto = load_panel(['BTC-USD','ETH-USD'], '2020-01-01', '2024-12-31')
```
Same code pattern, any asset class. *That's* the value of wrapping logic in functions.

---
## Summary Table

| Step | Code |
|------|------|
| Download equity | `yf.download(ticker, start, end, auto_adjust=True)` |
| Download macro | `web.DataReader('DGS10', 'fred', start, end)` |
| Inspect | `df.head()`, `df.info()`, `df.isnull().sum()` |
| Compute returns | `prices.pct_change().dropna()` |
| Style globally | `plt.rcParams.update({...})` once at the top |
| Annotate event | `ax.annotate('label', xy=..., xytext=..., arrowprops=...)` |
| Legend outside | `ax.legend(bbox_to_anchor=(0.5, 1.02))` |
| Reusable workflow | `load_panel(tickers, start, end)` |

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Simple Linear Regression — OLS Foundations & Diagnostics*